# Embeddings pour la détection d'un biais (lunettes) en utilisant la bibliothèque FiftyOne

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
from fiftyone import ViewField as F
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [3]:
dataset = fo.Dataset.from_dir(
    dataset_dir="data_full",
    dataset_type=fo.types.ImageDirectory,
)

 100% |███████████████| 7178/7178 [573.4ms elapsed, 0s remaining, 12.5K samples/s]      


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def predict_glasses(img_path):
    image = Image.open(img_path).convert("RGB")

    inputs = processor(
        text=["a person wearing glasses", "a person without glasses"],
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits_per_image.softmax(dim=1)

    return logits[0][0].item() > logits[0][1].item()

In [6]:
for sample in dataset:
    sample["glasses"] = predict_glasses(sample.filepath)
    sample.save()

In [7]:
print(dataset.count_values("glasses"))

{False: 6221, True: 957}


In [8]:
fob.compute_visualization(
    dataset,
    model="clip-vit-base32-torch",
    brain_key="embeddings"
)

Computing embeddings...


objc[6314]: Class AVFFrameReceiver is implemented in both /opt/miniconda3/envs/sam2-demo/lib/python3.10/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x12b7203a8) and /opt/miniconda3/envs/sam2-demo/lib/python3.10/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x14aa903a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[6312]: Class AVFFrameReceiver is implemented in both /opt/miniconda3/envs/sam2-demo/lib/python3.10/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x11925c3a8) and /opt/miniconda3/envs/sam2-demo/lib/python3.10/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x1398c43a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[6314]: Class AVFAudioReceiver is implemented in both /opt/miniconda3/envs/sam2-demo/lib/python3.10/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x12b7203f8) and /opt/miniconda3/env

 100% |███████████████| 7178/7178 [3.8m elapsed, 0s remaining, 28.4 samples/s]      
Generating visualization...
UMAP( verbose=True)
Wed Apr 15 15:40:42 2026 Construct fuzzy simplicial set
Wed Apr 15 15:40:42 2026 Finding Nearest Neighbors
Wed Apr 15 15:40:42 2026 Building RP forest with 9 trees
Wed Apr 15 15:40:44 2026 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	Stopping threshold met -- exiting after 4 iterations
Wed Apr 15 15:40:48 2026 Finished Nearest Neighbor Search
Wed Apr 15 15:40:49 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Wed Apr 15 15:40:51 2026 Finished embedding


In [11]:
session = fo.launch_app(dataset)
session.wait()

Notebook sessions cannot wait
